In [ ]:
%pip install --no-cache-dir --force-reinstall "numpy==1.26.4" "pyarrow==17.0.0" "datasets==3.6.0" "opencv-python==4.11.0.86" "mediapipe==0.10.21" scikit-learn pandas matplotlib joblib pillow
print("Installation complete. Restart the Jupyter kernel before running the next cell.")

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
import math
import time
import random
import io
from pathlib import Path
from collections import deque, Counter
from PIL import Image
from datasets import load_dataset, Image as HFImage
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

/opt/anaconda3/envs/marvel-magic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATASET_ID = "ntsrigaud/hagrid-subset"
BASE_DIR = Path("mystic_motion_project")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
FEATURE_COLUMNS = [f"{axis}_{i}" for i in range(21) for axis in ("x", "y", "z")]
POSE_LABELS = ["OPEN_PALM", "FIST", "POINT", "TWO_FINGERS", "PINCH"]
MODEL_PATH = MODEL_DIR / "hagrid_hand_pose_random_forest.joblib"
FEATURE_CACHE = DATA_DIR / "hagrid_hand_landmarks.csv"
MAX_IMAGES_PER_CLASS = 1500
MIN_DETECTIONS_PER_CLASS = 20
CONFIDENCE_THRESHOLD = 0.48

In [ ]:
CHARACTERS = {
    "1": "WANDA",
    "2": "DOCTOR STRANGE",
    "3": "THOR",
    "4": "LOKI",
    "5": "FIRE MAGIC",
    "6": "PURPLE COSMIC",
    "7": "SHANG-CHI",
    "8": "THANOS",
    "9": "ANCIENT ONE",
}
PALETTES = {
    "WANDA": (45, 35, 245),
    "DOCTOR STRANGE": (35, 155, 255),
    "THOR": (255, 225, 100),
    "LOKI": (70, 220, 95),
    "FIRE MAGIC": (25, 125, 255),
    "PURPLE COSMIC": (220, 70, 205),
    "SHANG-CHI": (30, 195, 255),
    "THANOS": (190, 65, 165),
    "ANCIENT ONE": (185, 220, 80),
}

In [ ]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

In [ ]:
print("OpenCV:", cv2.__version__, "| MediaPipe:", mp.__version__, "| Dataset:", DATASET_ID)

OpenCV: 4.11.0 | MediaPipe: 0.10.21 | Dataset: ntsrigaud/hagrid-subset


In [ ]:
import sys
import numpy as np
import pyarrow as pa
import datasets
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("PyArrow:", pa.__version__)
print("Hugging Face Datasets:", datasets.__version__)
assert np.__version__.startswith("1.26."), "Wrong NumPy version: restart kernel and verify the active environment."
assert pa.__version__.startswith("17.0."), "Wrong PyArrow version: reinstall in this kernel environment."
print("NumPy/PyArrow compatibility imports passed.")

Python: /opt/anaconda3/envs/marvel-magic/bin/python
NumPy: 1.26.4
PyArrow: 17.0.0
Hugging Face Datasets: 3.6.0
NumPy/PyArrow compatibility imports passed.


In [ ]:
def landmark_array(hand_landmarks):
    return np.asarray([(p.x, p.y, p.z) for p in hand_landmarks.landmark], dtype=np.float32)

In [ ]:
def normalized_features(points):
    points = np.asarray(points, dtype=np.float32).copy()
    if points.shape != (21, 3) or not np.isfinite(points).all():
        return None
    points -= points[0]
    scale = np.max(np.linalg.norm(points[:, :2], axis=1))
    if not np.isfinite(scale) or scale < 1e-6:
        return None
    points /= scale
    return points.reshape(-1)

In [ ]:
def pixel_point(points, index, width, height):
    return (int(np.clip(points[index, 0] * width, 0, width - 1)),
            int(np.clip(points[index, 1] * height, 0, height - 1)))

In [ ]:
def distance2d(a, b):
    return float(np.linalg.norm(np.asarray(a[:2]) - np.asarray(b[:2])))

In [ ]:
def finger_states(points):
    states = []
    for tip, pip, mcp in ((8, 6, 5), (12, 10, 9), (16, 14, 13), (20, 18, 17)):
        states.append(bool(distance2d(points[tip], points[0]) > distance2d(points[mcp], points[0]) * 1.18))
    thumb_extended = distance2d(points[4], points[5]) > distance2d(points[3], points[5]) * 1.08
    return [thumb_extended] + states

In [ ]:
class HandTracker:
    def __init__(self, max_hands=2, static_image_mode=False):
        self.detector = mp_hands.Hands(
            static_image_mode=static_image_mode, max_num_hands=max_hands,
            model_complexity=1, min_detection_confidence=0.55,
            min_tracking_confidence=0.55
        )

    def process(self, frame):
        result = self.detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        detected = []
        if result.multi_hand_landmarks:
            for index, landmarks in enumerate(result.multi_hand_landmarks):
                handedness = "Unknown"
                if result.multi_handedness and index < len(result.multi_handedness):
                    handedness = result.multi_handedness[index].classification[0].label
                if handedness == "Unknown":
                    handedness = f"Unknown-{index}"
                detected.append({"points": landmark_array(landmarks), "handedness": handedness,
                                 "landmarks": landmarks})
        return detected

    def close(self):
        self.detector.close()

print("Landmark extraction and live tracking are ready.")

Landmark extraction and live tracking are ready.


In [ ]:
dataset = load_dataset(DATASET_ID)
print("Dataset:", dataset)
print("Available splits:", list(dataset.keys()))

In [ ]:
for split_name, split_data in dataset.items():
    print(f"\\nSplit: {split_name} | rows: {len(split_data)}")
    print("Features:", split_data.features)
    if len(split_data):
        sample = split_data[0]
        print("Sample fields:", list(sample.keys()))
        for field, value in sample.items():
            if field.lower() in ("label", "labels", "gesture", "class", "category"):
                print("Label field/sample:", field, value)

In [ ]:
LABEL_FIELD_CANDIDATES = ("label", "labels", "gesture", "class", "category")
IMAGE_FIELD_CANDIDATES = ("image", "img", "pixel_values")
LABEL_ALIASES = {
    "palm": "OPEN_PALM",
    "open_palm": "OPEN_PALM",
    "fist": "FIST",
    "one": "POINT",
    "point": "POINT",
    "pointing": "POINT",
    "pointing_finger": "POINT",
    "peace": "TWO_FINGERS",
    "two_up": "TWO_FINGERS",
    "two_fingers": "TWO_FINGERS",
    "ok": "PINCH",
    "pinch": "PINCH",
}

In [ ]:
def raw_label_name(value, schema=None):
    if isinstance(value, (int, np.integer)) and schema is not None:
        try:
            value = schema.int2str(int(value))
        except Exception:
            return str(value)
    if isinstance(value, dict):
        value = value.get("name", value.get("label", value.get("id", value)))
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")

In [ ]:
def resolve_label(value, schema=None):
    return LABEL_ALIASES.get(raw_label_name(value, schema))

In [ ]:
def decode_image(value):
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(io.BytesIO(value["bytes"])).convert("RGB")
        if value.get("path"):
            return Image.open(value["path"]).convert("RGB")
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, np.ndarray):
        return Image.fromarray(value).convert("RGB")
    if isinstance(value, bytes):
        return Image.open(io.BytesIO(value)).convert("RGB")
    if isinstance(value, str):
        return Image.open(value).convert("RGB")
    raise TypeError(f"Unsupported image value: {type(value)}")

In [ ]:
observed_labels = Counter()
mapped_label_counts = Counter()
unmapped_labels = Counter()
failed_images = 0
failed_detections = 0
feature_rows = []
image_tracker = HandTracker(max_hands=2, static_image_mode=True)

try:
    for split_name, split_data in dataset.items():
        label_field = next((k for k in LABEL_FIELD_CANDIDATES if k in split_data.column_names), None)
        image_field = next((k for k in IMAGE_FIELD_CANDIDATES if k in split_data.column_names), None)
        if label_field is None or image_field is None:
            print(f"Skipping {split_name}: no recognized image/label fields.")
            continue
        schema = split_data.features.get(label_field)
        print(f"Extracting {split_name}: image={image_field}, label={label_field}")
        for row_index, example in enumerate(split_data):
            raw_value = example[label_field]
            raw_name = raw_label_name(raw_value, schema)
            observed_labels[raw_name] += 1
            target = LABEL_ALIASES.get(raw_name)
            if target is None:
                unmapped_labels[raw_name] += 1
                continue
            if mapped_label_counts[target] >= MAX_IMAGES_PER_CLASS:
                continue
            try:
                rgb = decode_image(example[image_field])
                bgr = cv2.cvtColor(np.asarray(rgb), cv2.COLOR_RGB2BGR)
                found = image_tracker.process(bgr)
                if not found:
                    failed_detections += 1
                    continue
                image_group = f"{split_name}:{row_index}"
                for hand in found:
                    vector = normalized_features(hand["points"])
                    if vector is None:
                        failed_detections += 1
                        continue
                    record = {"label": target, "source_split": split_name,
                              "source_group": image_group, "source_label": raw_name}
                    record.update(dict(zip(FEATURE_COLUMNS, vector.astype(float))))
                    feature_rows.append(record)
                mapped_label_counts[target] += 1
            except Exception:
                failed_images += 1
        print("Mapped image counts so far:", dict(mapped_label_counts))
finally:
    image_tracker.close()

In [ ]:
print("Observed dataset labels:", dict(observed_labels))
print("Mapped target image counts:", dict(mapped_label_counts))
print("Unsupported/unmapped labels:", dict(unmapped_labels))
print("Failed landmark detections:", failed_detections)
print("Failed image decoding/processing:", failed_images)

In [ ]:
if not feature_rows:
    raise RuntimeError("No mapped hand landmarks were extracted. Inspect the printed schema and labels; do not invent mappings.")
pose_df = pd.DataFrame(feature_rows)
pose_df.to_csv(FEATURE_CACHE, index=False)
print("Feature cache:", FEATURE_CACHE.resolve(), "| rows:", len(pose_df))
print(pose_df["label"].value_counts())

In [ ]:
pose_df = pd.read_csv(FEATURE_CACHE)
required = ["label", "source_split", "source_group"] + FEATURE_COLUMNS
missing = [column for column in required if column not in pose_df.columns]
if missing:
    raise ValueError(f"Feature cache missing columns: {missing}")

In [ ]:
pose_df = pose_df.dropna(subset=required).copy()
counts = pose_df["label"].value_counts()
usable_classes = [label for label in POSE_LABELS if counts.get(label, 0) >= MIN_DETECTIONS_PER_CLASS]
pose_df = pose_df[pose_df["label"].isin(usable_classes)].copy()
if len(usable_classes) < 2:
    raise ValueError(f"At least two target classes need {MIN_DETECTIONS_PER_CLASS} detections; available: {counts.to_dict()}")

In [ ]:
X = pose_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y = pose_df["label"].astype(str).to_numpy()
groups = pose_df["source_group"].astype(str).to_numpy()
split_names = set(pose_df["source_split"].astype(str))

In [ ]:
if {"train", "test"}.issubset(split_names):
    train_mask = pose_df["source_split"].astype(str).to_numpy() == "train"
    test_mask = pose_df["source_split"].astype(str).to_numpy() == "test"
    if len(set(y[train_mask])) >= 2 and len(set(y[test_mask])) >= 2:
        X_train, X_test, y_train, y_test = X[train_mask], X[test_mask], y[train_mask], y[test_mask]
        split_strategy = "provided train/test split"
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
        train_idx, test_idx = next(splitter.split(X, y, groups))
        X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
        split_strategy = "group holdout fallback"
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
    split_strategy = "group-level holdout by source image"

if len(set(y_train)) < 2 or len(set(y_test)) < 2:
    raise ValueError("The chosen split does not contain at least two classes in both partitions. Increase the extraction cap or inspect class coverage.")

In [ ]:
model = RandomForestClassifier(
    n_estimators=450, class_weight="balanced_subsample", min_samples_leaf=2,
    max_features="sqrt", random_state=SEED, n_jobs=-1
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
test_accuracy = accuracy_score(y_test, predictions)
print("Split strategy:", split_strategy)
print("Train rows:", len(y_train), "| Test rows:", len(y_test))
print("Held-out accuracy (measured):", round(float(test_accuracy), 4))

In [ ]:
print(classification_report(y_test, predictions, labels=usable_classes, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=usable_classes, xticks_rotation=35)
plt.title("MYSTIC MOTION — HaGRID Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
model_bundle = {
    "model": model, "feature_columns": FEATURE_COLUMNS, "classes": list(model.classes_),
    "dataset_id": DATASET_ID, "test_accuracy": float(test_accuracy),
    "label_aliases": LABEL_ALIASES, "normalization": "wrist-origin, max-XY-radius scale, 21x3"
}
joblib.dump(model_bundle, MODEL_PATH)
print("Saved model bundle:", MODEL_PATH.resolve())

In [ ]:
trained_bundle = joblib.load(MODEL_PATH)
live_pose_model = trained_bundle["model"]

In [ ]:
model_bundle = {
    "model": model, "feature_columns": FEATURE_COLUMNS, "classes": list(model.classes_),
    "dataset_id": DATASET_ID, "test_accuracy": float(test_accuracy),
    "label_aliases": LABEL_ALIASES, "normalization": "wrist-origin, max-XY-radius scale, 21x3"
}

joblib.dump(model_bundle, MODEL_PATH)
print("Saved model bundle:", MODEL_PATH.resolve())

In [ ]:
class LiveHandSmoother:
    def __init__(self, alpha=0.42, history_size=5):
        self.alpha = alpha
        self.points = {}
        self.poses = {}

    def update(self, hand, pose):
        key = hand["handedness"]
        current = hand["points"]
        if key in self.points:
            current = self.alpha * current + (1.0 - self.alpha) * self.points[key]
        self.points[key] = current
        hand["points"] = current
        history = self.poses.setdefault(key, deque(maxlen=5))
        history.append(pose)
        stable = Counter(history).most_common(1)[0][0]
        return hand, stable

    def retain(self, visible):
        for key in list(self.points):
            if key not in visible:
                self.points.pop(key, None)
                self.poses.pop(key, None)

In [ ]:
def predict_gesture(hand, model, threshold=CONFIDENCE_THRESHOLD):
    vector = normalized_features(hand["points"])
    if vector is None:
        return "UNKNOWN", 0.0
    probabilities = model.predict_proba(vector.reshape(1, -1))[0]
    index = int(np.argmax(probabilities))
    confidence = float(probabilities[index])
    if confidence < threshold:
        return "UNKNOWN", confidence
    return str(model.classes_[index]), confidence

In [ ]:
def hand_anchors(hand, width, height, gesture):
    p = hand["points"]
    palm = np.mean(p[[0, 5, 9, 13, 17], :2], axis=0)
    if gesture == "PINCH":
        return [((p[4, :2] + p[8, :2]) / 2, "pinch")]
    if gesture == "TWO_FINGERS":
        return [(p[8, :2], "index"), (p[12, :2], "middle"),
                ((p[8, :2] + p[12, :2]) / 2, "between")]
    if gesture == "POINT":
        return [(p[8, :2], "index"), (p[6, :2], "joint")]
    if gesture == "OPEN_PALM":
        return [(p[i, :2], "tip") for i in (4, 8, 12, 16, 20)] + [(palm, "palm")]
    if gesture == "FIST":
        return [(palm, "fist"), (p[0, :2], "wrist")]
    return [(palm, "palm")]

In [ ]:
def px(point, width, height):
    return int(point[0] * width), int(point[1] * height)

def alpha_composite(frame, overlay, alpha=0.72):
    cv2.addWeighted(overlay, alpha, frame, 1.0 - alpha, 0, frame)

def glow_dot(layer, center, radius, color):
    cv2.circle(layer, center, radius * 3, color, -1, cv2.LINE_AA)
    cv2.circle(layer, center, radius, (245, 250, 255), -1, cv2.LINE_AA)

def draw_ring(layer, center, radius, color, t, count=3, tilt=0):
    cx, cy = center
    for i in range(count):
        r = max(8, int(radius * (0.62 + 0.22 * i) + 4 * math.sin(t * 2.4 + i)))
        angle = (t * (30 + i * 9) + i * 37 + tilt) % 360
        cv2.ellipse(layer, (cx, cy), (r, max(5, int(r * (0.42 + 0.12 * i)))),
                    int(angle), 0, 340, color, 2, cv2.LINE_AA)

In [ ]:
def draw_arc(layer, a, b, color, t, branches=2):
    ax, ay = a
    bx, by = b
    mid = np.array([(ax + bx) / 2, (ay + by) / 2], dtype=float)
    direction = np.array([-(by - ay), bx - ax], dtype=float)
    norm = np.linalg.norm(direction) + 1e-6
    direction /= norm
    pts = [a]
    for j in range(1, 7):
        base = np.array(a, dtype=float) + (np.array(b, dtype=float) - a) * j / 7
        jitter = math.sin(t * 18 + j * 4.1) * (5 + branches * 2)
        q = base + direction * jitter
        pts.append(tuple(q.astype(int)))
    pts.append(b)
    cv2.polylines(layer, [np.asarray(pts, np.int32)], False, color, 2, cv2.LINE_AA)
    cv2.polylines(layer, [np.asarray(pts, np.int32)], False, (245, 250, 255), 1, cv2.LINE_AA)

In [ ]:
def draw_lightning(layer, hand, gesture, t, width, height):
    p = hand["points"]
    if gesture == "POINT":
        path = [px(p[i, :2], width, height) for i in (5, 6, 7, 8)]
        for a, b in zip(path[:-1], path[1:]):
            draw_arc(layer, a, b, (255, 220, 105), t, 3)
        draw_ring(layer, path[-1], 23, (255, 235, 140), t, 3)
        for k in range(3):
            orbit = (t * 3.4 + k * 2.1)
            q = (int(path[-1][0] + 15 * math.cos(orbit)), int(path[-1][1] + 15 * math.sin(orbit)))
            draw_arc(layer, path[-1], q, (255, 255, 220), t + k, 2)
    elif gesture == "TWO_FINGERS":
        a, b = px(p[8, :2], width, height), px(p[12, :2], width, height)
        draw_arc(layer, a, b, (255, 245, 180), t, 4)
        draw_ring(layer, a, 20, (255, 220, 100), t, 2)
        draw_ring(layer, b, 20, (255, 220, 100), -t, 2)
    elif gesture == "OPEN_PALM":
        tips = [px(p[i, :2], width, height) for i in (4, 8, 12, 16, 20)]
        palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
        draw_ring(layer, palm, 48, (255, 215, 85), t, 4)
        for tip in tips:
            draw_arc(layer, palm, tip, (255, 245, 190), t, 4)
    elif gesture == "FIST":
        palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
        draw_ring(layer, palm, 43, (255, 215, 80), t, 5)
        for i in range(12):
            angle = t * 2.8 + i * math.tau / 12
            q = (int(palm[0] + 38 * math.cos(angle)), int(palm[1] + 30 * math.sin(angle)))
            draw_arc(layer, palm, q, (255, 255, 225), t + i, 2)
    elif gesture == "PINCH":
        center = px((p[4, :2] + p[8, :2]) / 2, width, height)
        draw_ring(layer, center, 26, (255, 230, 105), t, 4)
        for i in range(16):
            angle = t * 2.5 + i * math.tau / 16
            q = (int(center[0] + 18 * math.cos(angle)), int(center[1] + 18 * math.sin(angle)))
            glow_dot(layer, q, 2, (255, 255, 245))

In [ ]:
def draw_character_gesture(frame, character, gesture, hand, t, width, height):
    overlay = frame.copy()
    p = hand["points"]
    palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
    anchors = hand_anchors(hand, width, height, gesture)
    points = [(px(a, width, height), kind) for a, kind in anchors]
    color = PALETTES[character]
    if character == "THOR":
        draw_lightning(overlay, hand, gesture, t, width, height)
    else:
        for center, kind in points:
            x, y = center
            if character == "WANDA":
                draw_ring(overlay, center, 38 if gesture == "PINCH" else 52, color, t, 4)
                for i in range(8):
                    angle = t * 1.7 + i * math.tau / 8
                    q = (int(x + 35 * math.cos(angle)), int(y + 28 * math.sin(angle)))
                    draw_arc(overlay, center, q, (90, 65, 255), t + i, 2)
            elif character == "DOCTOR STRANGE":
                draw_ring(overlay, center, 58 if gesture == "OPEN_PALM" else 36, color, t, 5)
                for i in range(12):
                    a = t * 0.55 + i * math.tau / 12
                    q = (int(x + 48 * math.cos(a)), int(y + 48 * math.sin(a)))
                    cv2.circle(overlay, q, 3, (100, 205, 255), -1, cv2.LINE_AA)
            elif character == "LOKI":
                for i in range(3):
                    pts = []
                    for j in range(30):
                        a = j * math.tau / 29 + t * (1 if i % 2 else -1) + i * 2.1
                        r = 18 + j * 1.25
                        pts.append((int(x + math.cos(a) * r), int(y + math.sin(a) * r * 0.55)))
                    cv2.polylines(overlay, [np.asarray(pts, np.int32)], False, color, 2, cv2.LINE_AA)
            elif character == "FIRE MAGIC":
                for i in range(22):
                    a = i * 2.399 + t * 1.8
                    rise = (i * 11 + int(t * 65)) % (75 if gesture == "PINCH" else 115)
                    q = (int(x + math.cos(a) * (rise * 0.36 + 5)), int(y - rise))
                    glow_dot(overlay, q, max(2, 7 - rise // 20), color)
            elif character in ("PURPLE COSMIC", "THANOS"):
                draw_ring(overlay, center, 38 if gesture == "PINCH" else 60, color, t, 5)
                for i in range(18):
                    a = t * (1.4 if character == "THANOS" else 1.9) + i * math.tau / 18
                    r = 28 + 15 * math.sin(t * 2 + i)
                    q = (int(x + r * math.cos(a)), int(y + r * 0.65 * math.sin(a)))
                    glow_dot(overlay, q, 2 if character == "THANOS" else 3, color)
            elif character == "SHANG-CHI":
                draw_ring(overlay, center, 45 if gesture == "FIST" else 58, color, t, 6)
                for i in range(3):
                    a = t * 1.3 + i * math.tau / 3
                    q = (int(x + 50 * math.cos(a)), int(y + 25 * math.sin(a)))
                    draw_arc(overlay, center, q, (120, 220, 255), t, 2)
            elif character == "ANCIENT ONE":
                draw_ring(overlay, center, 65 if gesture == "OPEN_PALM" else 42, color, t, 6)
                for i in range(16):
                    a = t * 0.7 + i * math.tau / 16
                    q = (int(x + 55 * math.cos(a)), int(y + 55 * math.sin(a)))
                    cv2.circle(overlay, q, 3, (220, 245, 150), -1, cv2.LINE_AA)
    if gesture == "FIST":
        draw_ring(overlay, palm, 52, color, -t, 2)
    elif gesture == "OPEN_PALM":
        draw_ring(overlay, palm, 68, color, t, 2)
    elif gesture == "PINCH":
        glow_dot(overlay, px((p[4, :2] + p[8, :2]) / 2, width, height), 7, color)
    alpha_composite(frame, overlay, 0.66)

In [ ]:
def draw_two_hand_circle(frame, center, radius, character, t):
    overlay = frame.copy()
    color = PALETTES[character]
    c = tuple(map(int, center))
    for i in range(5):
        r = int(radius * (0.48 + i * 0.13) + 5 * math.sin(t * 2 + i))
        cv2.circle(overlay, c, r, color if i % 2 else (235, 245, 255), 2, cv2.LINE_AA)
    for i in range(24):
        a = t * (0.35 if i % 2 else -0.45) + i * math.tau / 24
        inner = radius * 0.42
        outer = radius * (0.88 if i % 2 else 0.72)
        p1 = (int(c[0] + inner * math.cos(a)), int(c[1] + inner * math.sin(a)))
        p2 = (int(c[0] + outer * math.cos(a)), int(c[1] + outer * math.sin(a)))
        cv2.line(overlay, p1, p2, color, 2, cv2.LINE_AA)
        cv2.circle(overlay, p2, 4, (245, 250, 255), -1, cv2.LINE_AA)
    for i in range(8):
        a = -t * 0.7 + i * math.tau / 8
        q = (int(c[0] + radius * 0.58 * math.cos(a)), int(c[1] + radius * 0.58 * math.sin(a)))
        cv2.circle(overlay, q, max(3, radius // 35), color, -1, cv2.LINE_AA)
    alpha_composite(frame, overlay, 0.68)

print("Gesture-aware effects, nine character palettes, and two-hand magic are ready.")

In [ ]:
def run_mystic_motion(camera_index=0, confidence_threshold=CONFIDENCE_THRESHOLD):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError("Cannot open webcam. Check permissions or try camera_index=1.")
    tracker = HandTracker(max_hands=2)
    smoother = LiveHandSmoother()
    character = "THOR"
    show_skeleton = True
    show_help = True
    start_time = time.time()
    frame_failures = 0
    print("MYSTIC MOTION running: 1-9 select character | Q quit | S landmarks | T instructions")
    try:
        while True:
            ok, frame = cap.read()
            if not ok or frame is None:
                frame_failures += 1
                if frame_failures >= 10:
                    print("Camera repeatedly failed to provide frames; stopping.")
                    break
                time.sleep(0.03)
                continue
            frame_failures = 0
            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            elapsed = time.time() - start_time
            hands = tracker.process(frame)
            predictions = []
            visible = set()
            for hand in hands:
                key = hand["handedness"]
                visible.add(key)
                raw_pose, confidence = predict_gesture(hand, live_pose_model, confidence_threshold)
                hand, pose = smoother.update(hand, raw_pose)
                predictions.append((hand, pose, confidence))
                if show_skeleton:
                    mp_draw.draw_landmarks(frame, hand["landmarks"], mp_hands.HAND_CONNECTIONS)
            smoother.retain(visible)

            active, midpoint, radius, separation = two_hand_opposed_circle(hands)
            if active:
                center = (int(midpoint[0] * width), int(midpoint[1] * height))
                draw_two_hand_circle(frame, center, radius, character, elapsed)
                cv2.putText(frame, "TWO-HAND MAGIC", (center[0] - 90, center[1] + radius + 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (245, 245, 255), 2)
            else:
                for hand, gesture, confidence in predictions:
                    if gesture in POSE_LABELS:
                        draw_character_gesture(frame, character, gesture, hand, elapsed, width, height)

            primary = predictions[0] if predictions else None
            gesture_text = primary[1] if primary else "NO HAND"
            conf_text = primary[2] if primary else 0.0
            cv2.rectangle(frame, (0, 0), (width, 92), (12, 12, 25), -1)
            cv2.putText(frame, f"MYSTIC MOTION | {character}", (14, 31),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, (150, 225, 255), 2)
            cv2.putText(frame, f"Gesture: {gesture_text} | Confidence: {conf_text:.2f} | Hands: {len(hands)}",
                        (14, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)
            if len(predictions) > 1:
                second = predictions[1]
                cv2.putText(frame, f"Second hand: {second[1]} ({second[2]:.2f})",
                            (width - 300, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180, 230, 255), 1)
            if show_help:
                cv2.putText(frame, "1 Wanda  2 Strange  3 Thor  4 Loki  5 Fire  6 Cosmic  7 Shang-Chi  8 Thanos  9 Ancient",
                            (14, height - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 230, 255), 1)
            cv2.imshow("MYSTIC MOTION - AR Magic Simulator", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("s"):
                show_skeleton = not show_skeleton
            elif key == ord("t"):
                show_help = not show_help
            elif 49 <= key <= 57:
                character = CHARACTERS[chr(key)]
                print("Selected character:", character)
    finally:
        cap.release()
        tracker.close()
        cv2.destroyAllWindows()

run_mystic_motion(camera_index=0)

In [ ]:
def two_hand_opposed_circle(hands):
    if len(hands) < 2:
        return False, None, None, None
    first, second = hands[0], hands[1]
    p1, p2 = first["points"], second["points"]
    v1 = p1[9, :2] - p1[0, :2]
    v2 = p2[9, :2] - p2[0, :2]
    denom = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    cosine = float(np.dot(v1, v2) / denom)
    c1 = np.mean(p1[[0, 5, 9, 13, 17], :2], axis=0)
    c2 = np.mean(p2[[0, 5, 9, 13, 17], :2], axis=0)
    separation = float(np.linalg.norm(c1 - c2))
    facing = cosine < -0.35
    open_pair = all(sum(finger_states(h["points"])) >= 4 for h in (first, second))
    active = facing and open_pair and 0.12 < separation < 0.85
    return active, (c1 + c2) / 2.0, max(70, int(separation * 460)), separation